## Cell 1 — Environment and Imports

Loads all required libraries and Layer 1 artifacts (scaler, SVM model) as read-only.
Defines directory paths for MIMIC PERform AF CSV files.

**src/ calls:** `src.mimic_perform_af_features.build_mimic_feature_matrix`
**Layer 1 artifacts:** `outputs/models/scaler.joblib`, `outputs/models/selected_model.joblib` (read-only)
**Input directories:** `data/mimic_perform_af/mimic_perform_af_csv/`, `data/mimic_perform_af/mimic_perform_non_af_csv/`

In [ ]:
# src calls: src.mimic_perform_af_features.build_mimic_feature_matrix
# Layer 1 artifacts: outputs/models/scaler.joblib, outputs/models/selected_model.joblib (read-only)

import os
import numpy as np
import pandas as pd
import joblib
from scipy import stats
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, f1_score,
    recall_score, precision_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import neurokit2 as nk

import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
from mimic_perform_af_features import build_mimic_feature_matrix

# Confirm working directory
print("Working directory:", os.getcwd())

# Load Layer 1 artifacts (read-only — never overwrite)
scaler = joblib.load(os.path.join('outputs', 'models', 'scaler.joblib'))
model = joblib.load(os.path.join('outputs', 'models', 'selected_model.joblib'))

print("Scaler:", type(scaler).__name__)
print("Model:", type(model).__name__)

# Define paths
AF_DIR = os.path.join('data', 'mimic_perform_af', 'mimic_perform_af_csv')
NON_AF_DIR = os.path.join('data', 'mimic_perform_af', 'mimic_perform_non_af_csv')
LAYER2_OUT = os.path.join('outputs', 'layer2')

print(f"\nAF directory: {AF_DIR} — exists: {os.path.isdir(AF_DIR)}")
print(f"Non-AF directory: {NON_AF_DIR} — exists: {os.path.isdir(NON_AF_DIR)}")
print(f"Output directory: {LAYER2_OUT} — exists: {os.path.isdir(LAYER2_OUT)}")

## Cell 2 — Feature Matrix Construction

Runs the full MIMIC PERform AF pipeline: loads 35 CSV files, cleans PPG nulls via linear interpolation,
detects peaks with NeuroKit2 at 125 Hz, extracts RR intervals, computes the 8 locked features per subject.

Subjects with red quality tier (< 100 peaks) are skipped. Amber subjects (100–299 peaks) are flagged.

**src/ calls:** `src.mimic_perform_af_features.build_mimic_feature_matrix`
**Output:** `outputs/layer2/mimic_perform_af_features.csv`

In [ ]:
# src calls: src.mimic_perform_af_features.build_mimic_feature_matrix
# Output: outputs/layer2/mimic_perform_af_features.csv

features_df, failed_subjects = build_mimic_feature_matrix(AF_DIR, NON_AF_DIR)

# Assertions
feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']
assert features_df.shape[1] == 11, (
    f"Expected 11 columns, got {features_df.shape[1]}"
)
assert features_df[feature_cols].isnull().sum().sum() == 0, (
    "Null values found in feature columns"
)

# Save
features_df.to_csv(
    os.path.join(LAYER2_OUT, 'mimic_perform_af_features.csv'),
    index=False
)

# Summary
print(f"\nFeature matrix saved: {len(features_df)} subjects")
print(f"Failed subjects ({len(failed_subjects)}): {failed_subjects}")
print(f"\nClass distribution:")
print(features_df['label'].value_counts().sort_index().to_string())

## Cell 3 — Gap Quantification (KS Test)

Compares the distribution of each of the 8 features between Physionet Layer 1 training data
and MIMIC PERform AF Layer 2 data using the two-sample Kolmogorov-Smirnov test.

Classifies each feature's distributional distance: Large (KS >= 0.3), Moderate (0.1–0.3), Small (< 0.1).

**Input:** `data/processed/physionet_features.csv`, `features_df` from Cell 2
**Output:** `outputs/layer2/gap_quantification_mimic.csv`

In [ ]:
# src calls: none
# Input: data/processed/physionet_features.csv
# Output: outputs/layer2/gap_quantification_mimic.csv

physionet_df = pd.read_csv(
    os.path.join('data', 'processed', 'physionet_features.csv')
)

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

gap_results = []

for feat in feature_cols:
    physionet_vals = physionet_df[feat].dropna()
    mimic_vals = features_df[feat].dropna()

    ks_stat, p_val = stats.ks_2samp(physionet_vals, mimic_vals)

    # Determine direction
    if mimic_vals.median() > physionet_vals.median():
        direction = 'MIMIC higher'
    else:
        direction = 'MIMIC lower'

    # Classify magnitude
    if ks_stat >= 0.3:
        magnitude = 'Large'
    elif ks_stat >= 0.1:
        magnitude = 'Moderate'
    else:
        magnitude = 'Small'

    gap_results.append({
        'feature': feat,
        'ks_statistic': round(ks_stat, 4),
        'p_value': p_val,
        'direction': direction,
        'magnitude': magnitude
    })

gap_df = pd.DataFrame(gap_results)
gap_df.to_csv(
    os.path.join(LAYER2_OUT, 'gap_quantification_mimic.csv'),
    index=False
)

# Summary
large_count = (gap_df['magnitude'] == 'Large').sum()
moderate_count = (gap_df['magnitude'] == 'Moderate').sum()
small_count = (gap_df['magnitude'] == 'Small').sum()

print("Gap Quantification — KS Test Results")
print("=" * 55)
print(gap_df.to_string(index=False))
print(f"\nSummary: {large_count} large, {moderate_count} moderate, {small_count} small")

## Cell 4 — Model Application

Applies the Layer 1 SVM model to MIMIC PERform AF features. Uses `scaler.transform()` only (never
`fit_transform()`) and applies the fixed threshold of 0.34 to generate binary predictions.

No retraining, no threshold adjustment — strict Layer 1 to Layer 2 forward pass.

**Input:** `features_df` from Cell 2, `scaler` and `model` from Cell 1
**Output:** `outputs/layer2/probability_scores_mimic.csv`

In [ ]:
# src calls: none
# Output: outputs/layer2/probability_scores_mimic.csv

feature_cols = ['rmssd', 'sdnn', 'mean_rr', 'pnn50',
                'hr_mean', 'hr_std', 'rr_skewness', 'rr_kurtosis']

# Scale features — transform() only, never fit_transform()
X_mimic = features_df[feature_cols].values
X_mimic_scaled = scaler.transform(X_mimic)

# Get probability scores — Abnormal class (index 1)
prob_abnormal = model.predict_proba(X_mimic_scaled)[:, 1]

# Apply fixed threshold 0.34
THRESHOLD = 0.34
predicted_label = (prob_abnormal >= THRESHOLD).astype(int)

# Add to features_df
features_df['prob_abnormal'] = np.round(prob_abnormal, 6)
features_df['predicted_label'] = predicted_label

# Save
features_df.to_csv(
    os.path.join(LAYER2_OUT, 'probability_scores_mimic.csv'),
    index=False
)

# Summary
print("Model Application — SVM at threshold 0.34")
print("=" * 55)
print(f"Mean probability:    {prob_abnormal.mean():.4f}")
print(f"Median probability:  {np.median(prob_abnormal):.4f}")
print(f"Predicted Abnormal:  {(predicted_label == 1).sum()}")
print(f"Predicted Normal:    {(predicted_label == 0).sum()}")

## Cell 5 — Evaluation Against Ground Truth

Compares SVM predictions against pre-assigned AF/NSR labels.
Computes sensitivity, specificity, AUROC, F1, and confusion matrix.

Checks pre-registered criteria: sensitivity >= 80%, specificity >= 75%.
Reports amber-tier subjects separately if any exist.

In [ ]:
# src calls: none

y_true = features_df['label'].values
y_pred = features_df['predicted_label'].values
y_prob = features_df['prob_abnormal'].values

# Confusion matrix: labels [0, 1] → TN, FP, FN, TP
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
auroc = roc_auc_score(y_true, y_prob)
f1 = f1_score(y_true, y_pred)

# Pre-registered criteria
sens_pass = "PASS" if sensitivity >= 0.80 else "FAIL"
spec_pass = "PASS" if specificity >= 0.75 else "FAIL"

print("Evaluation Against Ground Truth — SVM at threshold 0.34")
print("=" * 55)
print(f"Sensitivity:   {sensitivity:.4f}  ({sensitivity*100:.1f}%)")
print(f"Specificity:   {specificity:.4f}  ({specificity*100:.1f}%)")
print(f"AUROC:         {auroc:.4f}")
print(f"F1 (Abnormal): {f1:.4f}")
print(f"\nConfusion Matrix:")
print(f"  TP={tp}  FP={fp}")
print(f"  FN={fn}  TN={tn}")
print(f"\nPre-registered criteria:")
print(f"  Sensitivity >= 80%: {sens_pass}")
print(f"  Specificity >= 75%: {spec_pass}")

# Amber subjects — separate reporting
amber_mask = features_df['quality_tier'] == 'amber'
if amber_mask.sum() > 0:
    amber_df = features_df[amber_mask]
    amber_y_true = amber_df['label'].values
    amber_y_pred = amber_df['predicted_label'].values
    amber_correct = (amber_y_true == amber_y_pred).sum()
    print(f"\nAmber-tier subjects ({amber_mask.sum()}):")
    for _, row in amber_df.iterrows():
        match = "correct" if row['label'] == row['predicted_label'] else "incorrect"
        print(f"  {row['subject_id']}: true={int(row['label'])}, "
              f"pred={int(row['predicted_label'])}, "
              f"prob={row['prob_abnormal']:.4f} — {match}")
    print(f"  Amber accuracy: {amber_correct}/{amber_mask.sum()}")
else:
    print("\nNo amber-tier subjects.")

## Cell 6 — Tier Assessment

Final tier evaluation for MIMIC PERform AF Layer 2 validation:
- **Tier 1:** Pipeline executed and gap quantification completed
- **Tier 2:** Sensitivity >= 80% AND specificity >= 75%
- **Tier 3:** AUROC >= 0.85

**Output:** `outputs/layer2/tier_assessment_mimic.csv`

In [ ]:
# src calls: none
# Output: outputs/layer2/tier_assessment_mimic.csv

# Tier 1: pipeline executed, gap quantification complete
tier_1 = "PASS" if len(features_df) > 0 else "FAIL"

# Tier 2: sensitivity >= 80% AND specificity >= 75%
tier_2 = "PASS" if (sensitivity >= 0.80 and specificity >= 0.75) else "FAIL"

# Tier 3: AUROC >= 0.85
tier_3 = "PASS" if auroc >= 0.85 else "FAIL"

tier_results = pd.DataFrame([
    {'tier': 'Tier 1 — Pipeline execution', 'result': tier_1},
    {'tier': 'Tier 2 — Sens >= 80% & Spec >= 75%', 'result': tier_2},
    {'tier': 'Tier 3 — AUROC >= 0.85', 'result': tier_3}
])

tier_results.to_csv(
    os.path.join(LAYER2_OUT, 'tier_assessment_mimic.csv'),
    index=False
)

print("Tier Assessment — MIMIC PERform AF")
print("=" * 55)
print(f"Tier 1 — Pipeline execution:          {tier_1}")
print(f"Tier 2 — Sens >= 80% & Spec >= 75%:   {tier_2}")
print(f"Tier 3 — AUROC >= 0.85:               {tier_3}")